# 01 – Delta Update

Inkrementelles Update der Delta Tables aus einem neuen Polar-Export-ZIP.

**Regelmässig ausführen** – z.B. nach jedem neuen Datenexport von Polar.

## Ablauf
1. Secrets & Verbindung prüfen
2. ZIP-Dateien in `input/` erkennen
3. MD5-Hashes gegen `import_log` prüfen (nur neue/geänderte Dateien)
4. DataFrames per MERGE INTO in Delta Tables schreiben
5. Import-Log aktualisieren
6. ZIP nach `archive/` verschieben
7. Delta-Bericht & Tabellenstand ausgeben

---
> 📦 **Vorbereitung:** Neues Polar-Export-ZIP in den `input/` Ordner legen,  
> dann alle Zellen von oben nach unten ausführen (`Run All`).

## 1 – Imports & Secrets prüfen

In [13]:
import os
import sys
from pathlib import Path
from datetime import datetime

# src/ zum Python-Pfad hinzufügen
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from db_loader import DatabricksLoader, secrets_pruefen
from delta_updater import DeltaUpdater

# ── Secrets prüfen ──────────────────────────────────────────
secrets_pruefen()

print(f"\nNotebook gestartet: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Alle Databricks Secrets gefunden.

Notebook gestartet: 2026-03-21 10:58:45


## 2 – ZIP-Dateien erkennen

In [14]:
input_ordner = Path('/workspaces/polar_databricks/input')
zip_dateien  = sorted(input_ordner.glob('*.zip'))

print(f"📂 Suche ZIP-Dateien in: {input_ordner.resolve()}")
print()

if not zip_dateien:
    print("⚠️  Keine ZIP-Dateien gefunden!")
    print()
    print("So exportierst du deine Polar-Daten:")
    print("  1. Polar Flow Web (flow.polar.com) oder Polar App öffnen")
    print("  2. Profil → Einstellungen → Eigene Daten exportieren")
    print("  3. ZIP herunterladen")
    print("  4. Datei in den 'input/' Ordner dieses Projekts kopieren")
    print("  5. Diese Zelle erneut ausführen")
else:
    print(f"Gefunden: {len(zip_dateien)} ZIP-Datei(en)")
    print()
    for z in zip_dateien:
        groesse_mb = z.stat().st_size / 1024 / 1024
        mtime      = datetime.fromtimestamp(z.stat().st_mtime)
        print(f"  📦 {z.name}")
        print(f"     Grösse   : {groesse_mb:.1f} MB")
        print(f"     Geändert : {mtime.strftime('%Y-%m-%d %H:%M')}")

📂 Suche ZIP-Dateien in: /workspaces/polar_databricks/input

Gefunden: 1 ZIP-Datei(en)

  📦 polar-user-data-export_31872feb-dd92-45be-a696-b0a56975877a.zip
     Grösse   : 539.9 MB
     Geändert : 2026-03-21 09:50


## 3 – Vorschau: Was ist neu?

Vergleicht die ZIP-Inhalte mit dem Import-Log, **ohne** Daten zu schreiben.

In [15]:
import hashlib
import zipfile
import re

if not zip_dateien:
    print("⏭️  Kein ZIP vorhanden – Zelle übersprungen")
else:
    # Import-Log laden
    db = DatabricksLoader()
    df_log = db.abfrage(
        f"SELECT dateiname, hash_md5, kategorie "
        f"FROM {db.catalog}.{db.schema}.import_log"
    )
    bekannte_hashes = dict(zip(df_log['dateiname'], df_log['hash_md5'])) \
                      if not df_log.empty else {}
    print(f"Import-Log: {len(bekannte_hashes)} bekannte Dateien")
    print()

    # Vorschau pro ZIP
    for zip_pfad in zip_dateien:
        print(f"── {zip_pfad.name} ──")
        zaehler = {'neu': 0, 'geaendert': 0, 'unveraendert': 0}
        kategorien_neu = {}

        with zipfile.ZipFile(zip_pfad, 'r') as zf:
            json_dateien = [n for n in zf.namelist() if n.endswith('.json')]

            for name in json_dateien:
                with zf.open(name) as f:
                    h = hashlib.md5(f.read()).hexdigest()

                if name not in bekannte_hashes:
                    zaehler['neu'] += 1
                    # Kategorie bestimmen
                    if   'activity_' in name: kat = 'activity'
                    elif 'training_' in name: kat = 'training'
                    elif '247ohr_'   in name: kat = 'heartrate'
                    elif 'ppi_'      in name: kat = 'hrv'
                    else:                     kat = 'sonstige'
                    kategorien_neu[kat] = kategorien_neu.get(kat, 0) + 1
                elif bekannte_hashes[name] != h:
                    zaehler['geaendert'] += 1
                else:
                    zaehler['unveraendert'] += 1

        print(f"  JSON-Dateien gesamt : {len(json_dateien)}")
        print(f"  Neu                 : {zaehler['neu']}")
        print(f"  Geändert            : {zaehler['geaendert']}")
        print(f"  Unverändert         : {zaehler['unveraendert']}")
        if kategorien_neu:
            print(f"  Neue Dateien nach Kategorie:")
            for kat, n in sorted(kategorien_neu.items()):
                print(f"    {kat:<12}: {n}")
        else:
            print("  ℹ️  Keine neuen Dateien – ZIP bereits vollständig importiert")
        print()

✅ Alle Databricks Secrets gefunden.
🔧 DatabricksLoader initialisiert
   Catalog : workspace
   Schema  : polar
   Host    : https://dbc-440507f2-1163.cloud.databricks.com
✅ Datenbankverbindung geschlossen
✅ Databricks verbunden
Import-Log: 4951 bekannte Dateien

── polar-user-data-export_31872feb-dd92-45be-a696-b0a56975877a.zip ──
  JSON-Dateien gesamt : 8075
  Neu                 : 3124
  Geändert            : 0
  Unverändert         : 4951
  Neue Dateien nach Kategorie:
    sonstige    : 3124



## 4 – Delta Update ausführen

> ✅ Nur ausführen wenn Schritt 3 neue/geänderte Dateien angezeigt hat.

In [16]:
if not zip_dateien:
    print("⏭️  Kein ZIP vorhanden – Zelle übersprungen")
else:
    updater  = DeltaUpdater()
    berichte = []

    for zip_pfad in zip_dateien:
        bericht = updater.verarbeite_zip(str(zip_pfad))
        berichte.append({'datei': zip_pfad.name, **bericht})


📦 Starte Delta-Update: polar-user-data-export_31872feb-dd92-45be-a696-b0a56975877a.zip
   Zeitpunkt: 2026-03-21 10:59:14
✅ Databricks verbunden: https://dbc-440507f2-1163.cloud.databricks.com

📋 Lade Import-Log...
   Bereits importiert: 4951 Dateien

🔍 Prüfe Dateien im ZIP...
   Neu         :   3124
   Geändert    :      0
   Unverändert :   4951
   Zu importieren: 3124

⚙️  Parse 3124 neue/geänderte Dateien...
📦 ZIP geladen: /workspaces/polar_databricks/input/polar-user-data-export_31872feb-dd92-45be-a696-b0a56975877a.zip
   Dateien gesamt :   8078
   activity_*     :   3751
   training_*     :   3124
   247ohr_*       :     60
   ppi_*          :     51

▶ Activity-Daten...
   Activity: 0/3751...
   Activity: 500/3751...
   Activity: 1000/3751...
   Activity: 1500/3751...
   Activity: 2000/3751...
   Activity: 2500/3751...
   Activity: 3000/3751...
   Activity: 3500/3751...
✅ Activity: 3751 Tage geladen (0 Fehler)
   ✅ activity: 3751 Datensätze verarbeitet

▶ Trainings-Daten...
   T

## 5 – Zusammenfassung

In [17]:
import pandas as pd

if not zip_dateien:
    print("⏭️  Kein ZIP verarbeitet")
else:
    # Bericht-Tabelle
    df_bericht = pd.DataFrame(berichte)
    print("📊 Import-Bericht:")
    display(df_bericht)

    # Gesamtzahlen
    gesamt_neu      = df_bericht['neu'].sum()
    gesamt_geaendert = df_bericht['geaendert'].sum()
    gesamt_dauer    = df_bericht['dauer_sekunden'].sum()
    print(f"\nGesamt neu importiert : {gesamt_neu}")
    print(f"Gesamt geändert       : {gesamt_geaendert}")
    print(f"Gesamtdauer           : {gesamt_dauer:.1f}s")

📊 Import-Bericht:


,datei,neu,geaendert,unveraendert,fehler,dauer_sekunden
0,polar-user-data-export_31872feb-dd92-45be-a696...,3124,0,4951,0,150.1



Gesamt neu importiert : 3124
Gesamt geändert       : 0
Gesamtdauer           : 150.1s


## 6 – Tabellenstand nach Update

In [18]:
# Aktuellen Stand aller Tabellen anzeigen
db2 = DatabricksLoader()
df_stand = db2.tabellen_uebersicht()
display(df_stand)

# Import-Log Übersicht
print("\n── Import-Log nach Update ──")
display(db2.import_log_uebersicht())

# Archiv prüfen
archiv_ordner = Path('/workspaces/polar_databricks/archive')
archiv_zips   = list(archiv_ordner.glob('*.zip'))
print(f"\n📁 Archiv: {len(archiv_zips)} ZIP-Datei(en)")
for z in sorted(archiv_zips):
    groesse_mb = z.stat().st_size / 1024 / 1024
    print(f"   {z.name}  ({groesse_mb:.1f} MB)")

db2.schliessen()
print("\n✅ Delta Update abgeschlossen – weiter mit 02_exploration.ipynb")

✅ Alle Databricks Secrets gefunden.
🔧 DatabricksLoader initialisiert
   Catalog : workspace
   Schema  : polar
   Host    : https://dbc-440507f2-1163.cloud.databricks.com
✅ Datenbankverbindung geschlossen
✅ Databricks verbunden

📊 Tabellen-Übersicht:
  tabelle  zeilen  min_datum  max_datum
 activity    3751 2014-04-25 2026-03-15
 training    3125 2014-03-24 2026-03-13
heartrate    1710 2018-08-31 2026-03-15
      hrv     369 2024-12-23 2026-03-15


,tabelle,zeilen,min_datum,max_datum
0,activity,3751,2014-04-25,2026-03-15
1,training,3125,2014-03-24,2026-03-13
2,heartrate,1710,2018-08-31,2026-03-15
3,hrv,369,2024-12-23,2026-03-15



── Import-Log nach Update ──
✅ Import-Log: 5 Kategorien


,kategorie,anzahl,letzter_import
0,activity,3751,2026-03-21 10:52:49.132282+00:00
1,training,3124,2026-03-21 10:59:42.089060+00:00
2,sonstige,1089,2026-03-21 10:53:04.617141+00:00
3,heartrate,60,2026-03-21 10:52:50.644176+00:00
4,hrv,51,2026-03-21 10:53:04.446052+00:00



📁 Archiv: 1 ZIP-Datei(en)
   polar-user-data-export_31872feb-dd92-45be-a696-b0a56975877a.zip  (539.9 MB)
✅ Datenbankverbindung geschlossen

✅ Delta Update abgeschlossen – weiter mit 02_exploration.ipynb
